In [4]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.metrics import r2_score

# ──────────────────────────────────────────────────────────────────────────────
# 1. LOAD DATA & PARSE TIME (Strict 82.21 Base)
# ──────────────────────────────────────────────────────────────────────────────
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

def parse_time(df: pd.DataFrame) -> pd.DataFrame:
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'] = parts[0]
    df['minute'] = parts[1]
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
    return df

train = parse_time(train)
test  = parse_time(test)

# ──────────────────────────────────────────────────────────────────────────────
# 2. LAG FEATURES (Strict 82.21 Base)
# ──────────────────────────────────────────────────────────────────────────────
demand_lookup = train.set_index(['geohash', 'day', 'time_slot'])['demand']

def add_lags(df: pd.DataFrame) -> pd.DataFrame:
    idx_1day = pd.MultiIndex.from_arrays([df['geohash'], df['day'] - 1, df['time_slot']])
    df['demand_lag_1day'] = demand_lookup.reindex(idx_1day).values
    
    slot_15m, day_15m = df['time_slot'] - 1, df['day']
    rollover_15m = slot_15m < 0
    day_15m = np.where(rollover_15m, day_15m - 1, day_15m)
    slot_15m = np.where(rollover_15m, slot_15m + 96, slot_15m)
    df['demand_lag_15min'] = demand_lookup.reindex(pd.MultiIndex.from_arrays([df['geohash'], day_15m, slot_15m])).values

    slot_1hr, day_1hr = df['time_slot'] - 4, df['day']
    rollover_1hr = slot_1hr < 0
    day_1hr = np.where(rollover_1hr, day_1hr - 1, day_1hr)
    slot_1hr = np.where(rollover_1hr, slot_1hr + 96, slot_1hr)
    df['demand_lag_1hr'] = demand_lookup.reindex(pd.MultiIndex.from_arrays([df['geohash'], day_1hr, slot_1hr])).values
    return df

train = add_lags(train)
test  = add_lags(test)

# ──────────────────────────────────────────────────────────────────────────────
# 3. IMPUTATION (Safe, historical only)
# ──────────────────────────────────────────────────────────────────────────────
ref = train[train['day'] == 48] 
geo_stats = ref.groupby('geohash')['demand'].agg(geo_mean='mean')
train = train.merge(geo_stats.reset_index(), on='geohash', how='left')
test  = test.merge(geo_stats.reset_index(), on='geohash', how='left')

global_mean = train['demand'].mean()
train['Temperature'] = train['Temperature'].fillna(-999)
test['Temperature']  = test['Temperature'].fillna(-999)

lag_cols = ['demand_lag_1day', 'demand_lag_15min', 'demand_lag_1hr']
for col in lag_cols:
    train[col] = train[col].fillna(train['geo_mean']).fillna(global_mean)
    test[col]  = test[col].fillna(test['geo_mean']).fillna(global_mean)

CAT_FEATURES = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in CAT_FEATURES:
    train[col] = train[col].fillna('Unknown').astype(str)
    test[col]  = test[col].fillna('Unknown').astype(str)
    
# LightGBM requires categoricals to be explicitly cast as 'category' dtype
for col in CAT_FEATURES:
    train[col] = train[col].astype('category')
    test[col]  = test[col].astype('category')

# ──────────────────────────────────────────────────────────────────────────────
# 4. SPLIT & FEATURES
# ──────────────────────────────────────────────────────────────────────────────
FEATURES = [
    'geohash', 'day', 'hour', 'minute', 'time_slot',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Temperature', 'Weather',
    'demand_lag_1day', 'demand_lag_15min', 'demand_lag_1hr'
]
TARGET = 'demand'

train_df = train[train['day'] == 48].copy()
val_df   = train[train['day'] == 49].copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val,   y_val   = val_df[FEATURES],   val_df[TARGET]

# ──────────────────────────────────────────────────────────────────────────────
# 5. MODEL 1: CATBOOST
# ──────────────────────────────────────────────────────────────────────────────
print("Training CatBoost...")
cb_model = CatBoostRegressor(iterations=3000, learning_rate=0.05, depth=6, random_seed=42, od_type='Iter', od_wait=150, verbose=0)
cb_model.fit(X_train, y_train, cat_features=CAT_FEATURES, eval_set=(X_val, y_val))
cb_val_preds = np.clip(cb_model.predict(X_val), 0, None)

# Retrain CatBoost on all data
best_cb = cb_model.best_iteration_
cb_final = CatBoostRegressor(iterations=best_cb, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
cb_final.fit(train[FEATURES], train[TARGET], cat_features=CAT_FEATURES)
cb_test_preds = np.clip(cb_final.predict(test[FEATURES]), 0, None)

# ──────────────────────────────────────────────────────────────────────────────
# 6. MODEL 2: LIGHTGBM
# ──────────────────────────────────────────────────────────────────────────────
print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=150, verbose=False)])
lgb_val_preds = np.clip(lgb_model.predict(X_val), 0, None)

# Retrain LightGBM on all data
best_lgb = lgb_model.best_iteration_
lgb_final = lgb.LGBMRegressor(n_estimators=best_lgb, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
lgb_final.fit(train[FEATURES], train[TARGET])
lgb_test_preds = np.clip(lgb_final.predict(test[FEATURES]), 0, None)

# ──────────────────────────────────────────────────────────────────────────────
# 7. BLEND & EVALUATE
# ──────────────────────────────────────────────────────────────────────────────
blended_val_preds = (cb_val_preds * 0.5) + (lgb_val_preds * 0.5)
r2 = r2_score(y_val, blended_val_preds)
print(f"\nBlended Validation R²: {r2:.5f} | Estimated Score: {max(0.0, 100.0 * r2):.2f}/100")

blended_test_preds = (cb_test_preds * 0.5) + (lgb_test_preds * 0.5)
submission = pd.DataFrame({'Index': test['Index'], 'demand': blended_test_preds})
submission.to_csv('submission_blended.csv', index=False)
print("Saved to submission_blended.csv")

Training CatBoost...
Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001546 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2227
[LightGBM] [Info] Number of data points in the train set: 69427, number of used features: 13
[LightGBM] [Info] Start training from score 0.092659
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.00